# BERTopic Modeling II
- 원본 데이터(WOS_dat.xls)를 다시 전처리(custom stopwords 추가 및 하이퍼파라미터 조정)한 output_02_cleaned_abstracts.csv로 2차 BERTopic Modeling
- 실행 환경: Colab T4(Python 3)

# [0] Colab 환경 설정

## 0-1. 구글드라이브 마운트

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 0-2. 패키지 설치
- 설치 후 런타임 재시작 및 모두 실행

In [2]:
!pip install bertopic
!pip install sentence-transformers
!pip install umap-learn
!pip install hdbscan


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 8.7 MB/s eta 0:00:00


## 0-3. 라이브러리 설정
약 1분 소요

In [3]:
import os
import pandas as pd
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.vectorizers import ClassTfidfTransformer
from umap import UMAP
from hdbscan import HDBSCAN

## 0-4. 파일 경로 설정 및 데이터 로드

In [4]:
# 기본 경로
base_path = '/content/drive/MyDrive/Colab Notebooks/GenAI-Finance-TopicModeling/'

# 파일 경로
file_path = os.path.join(
    base_path,
    'outputs',
    'output_02_cleaned_abstracts.csv'
)

# CSV 불러오기
article_df = pd.read_csv(file_path)

# 데이터 확인
print(article_df.shape)

article_df.head()

(602, 4)


,Title,Author,Abstract,Cleaned_Abstract
0,The Odyssey of robots.txt Governance: Measurin...,"Cui, J; Zha, MM; Wang, XF; Liao, XJ",Web content is an essential element for large ...,web content essential element model service su...
1,Evaluation of a Large Language Model on the Am...,"Ramgopal, S; Varma, S; Gorski, JK; Kester, KM;...","BackgroundLarge language models (LLMs), includ...",backgroundlarge model llm including chatgpt ch...
2,HDLGen-ChatGPT Case Study: RISC-V Processor VH...,"Morgan, F; Byrne, JP; Bupathi, A; George, R; E...",This paper presents the open source HDLGen-Cha...,present open source hdlgen chatgpt working tan...
3,Experience with Large Language Model Applicati...,"Yu, L; Alégroth, E; Chatzipetrou, P; Gorschek, T",Large Language Models (LLMs) offer promising c...,model llm offer promising capability informati...
4,Large Language Model Agents for Investment Man...,"Saha, P; Lyu, JR; Saxena, A; Zhao, TJ; Mehta, D",Recent advances in Large Language Models (LLMs...,recent advance model llm triggered new wave in...


# [1] BERTopic Modeling

## 1-1. 문서 리스트 생성

In [5]:
# BERTopic 입력 문서
docs = article_df['Cleaned_Abstract'].tolist()

print(f"문서 개수: {len(docs)}")

문서 개수: 602


## 1-2. 모델 설계

In [6]:
# SentenceTransformer 모델
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

# UMAP 모델: 차원 축소
umap_model = UMAP(
    n_neighbors=10,    # 15 -> 10으로 수정
    n_components=5,
    min_dist=0.0,
    metric='cosine',
    random_state=42
)

# HDBSCAN 모델: 클러스터링
hdbscan_model = HDBSCAN(
    min_cluster_size=8,      # 15 -> 8로 수정
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)

# CountVectorizer 모델: BoW 벡터화
vectorizer_model = CountVectorizer(
    stop_words='english',
    ngram_range=(1, 2),      # 5 -> 3으로 수정
)

# c-tf-idf 모델: 토픽별 중요 단어 계산
ctfidf_model = ClassTfidfTransformer()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## 1-3. BERTopic 모델 생성

In [7]:
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    nr_topics="auto",    # 8 -> auto 로 수정 (모델이 적절하다고 판단하는 토픽 개수 자동으로 결정하도록)
    calculate_probabilities=True,
    verbose=True
)

## 1-4. 모델 학습


In [8]:
topics, probs = topic_model.fit_transform(docs)

2026-05-21 03:53:58,355 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/19 [00:00<?, ?it/s]

2026-05-21 03:54:00,349 - BERTopic - Embedding - Completed ✓
2026-05-21 03:54:00,350 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-21 03:54:10,844 - BERTopic - Dimensionality - Completed ✓
2026-05-21 03:54:10,845 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-21 03:54:10,890 - BERTopic - Cluster - Completed ✓
2026-05-21 03:54:10,891 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-05-21 03:54:11,157 - BERTopic - Representation - Completed ✓
2026-05-21 03:54:11,158 - BERTopic - Topic reduction - Reducing number of topics
2026-05-21 03:54:11,167 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-21 03:54:11,430 - BERTopic - Representation - Completed ✓
2026-05-21 03:54:11,432 - BERTopic - Topic reduction - Reduced number of topics from 15 to 15


## 1-5. 결과 확인

### 토픽 결과 확인

In [9]:
print("[토픽 정보]")
topic_info = topic_model.get_topic_info()
topic_info

[토픽 정보]


,Topic,Count,Name,Representation,Representative_Docs
0,-1,160,-1_model_data_llm_domain,"[model, data, llm, domain, finance, financial,...",[traditional model nlp require considerable am...
1,0,110,0_generative_financial_data_technology,"[generative, financial, data, technology, fina...",[discusses transformative impact emerging digi...
2,1,92,1_question_chatgpt_medical_performance,"[question, chatgpt, medical, performance, accu...",[background artificial intelligence potential ...
3,2,41,2_sentiment_market_financial_model,"[sentiment, market, financial, model, trading,...",[financial sentiment analysis fsa aim infer se...
4,3,40,3_privacy_security_data_attack,"[privacy, security, data, attack, vulnerabilit...",[model llm widely adopted different domain inc...
5,4,22,4_contract_blockchain_defi_smart contract,"[contract, blockchain, defi, smart contract, s...",[rampant scam plague decentralized finance def...
6,5,20,5_retrieval_rag_domain_generation,"[retrieval, rag, domain, generation, augmented...",[rapid advancement intelligent retrieval techn...
7,6,19,6_llm_test_api_domain,"[llm, test, api, domain, tool, model, apis, pr...",[fintech software crucial safety timely market...
8,7,19,7_memory_pim_bank_dram,"[memory, pim, bank, dram, inference, processin...",[deploying model llm edge device pose signific...
9,8,15,8_extraction_narrative_entity_model,"[extraction, narrative, entity, model, text, d...",[story component namely event participant rela...


### 토픽 키워드 확인

In [10]:
for topic_num in topic_info['Topic']:
    if topic_num != -1:
        print(f"\n[Topic {topic_num}]")
        print(topic_model.get_topic(topic_num))


[Topic 0]
[('generative', np.float64(0.01698633485758705)), ('financial', np.float64(0.016769941916633356)), ('data', np.float64(0.015091077123159831)), ('technology', np.float64(0.01491022038090518)), ('finance', np.float64(0.013982399746447308)), ('model', np.float64(0.013843866830528849)), ('intelligence', np.float64(0.01205700067170034)), ('industry', np.float64(0.01190963013421434)), ('human', np.float64(0.010724637533315404)), ('artificial', np.float64(0.010596210441713683))]

[Topic 1]
[('question', np.float64(0.04467395921665182)), ('chatgpt', np.float64(0.033289151908881535)), ('medical', np.float64(0.02274977992941602)), ('performance', np.float64(0.021358652158875575)), ('accuracy', np.float64(0.01895244360392652)), ('clinical', np.float64(0.017022465518402172)), ('examination', np.float64(0.01650659335758051)), ('model', np.float64(0.01572162453355399)), ('llm', np.float64(0.014994310112651461)), ('answer', np.float64(0.014393538582439691))]

[Topic 2]
[('sentiment', np.fl

## 1-6. 결과 저장

In [17]:
topic_info.to_csv(
    os.path.join(
        base_path,
        'outputs',
        'output_03_topic_info.csv'
    ),
    index=False,
    encoding='utf-8-sig'
)

# [2] 시각화

## 2-1. Intertopic Distance Map

In [12]:
topic_model.visualize_topics()

## 2-2. Topic Word Scores

In [13]:
topic_model.visualize_barchart(
    top_n_topics=len(topic_info)-1
)

## 2-3. Topic Similarity Heatmap

In [14]:
topic_model.visualize_heatmap()